# 03 结构化输出与数据校验（重制版）

本文件是本课当前唯一讲义，保留已完成的课堂练习与理解题。旧版重复讲义已清理；作业文件为 `structured_ticket.py`，测试文件为 `test_structured_ticket.py`。

**重制原因**（诊断结论，见根目录《课程创建规范》第18.1节）：能够说出每个名词的定义，但说不清它们在什么时刻发生、顺序为什么这样，合上就散。这属于"关系型知识缺地图"，所以本版第一件事不是讲 JSON，而是给出全局地图与时间线，让每个知识点有明确的出场时刻。

三个业务对象的分工保持原样：讲义示例用图书，课堂练习用观影，课末作业用客服工单。四道课堂练习的函数名或类名与原讲义一致（`read_movie`、`MovieRecord`、`movie_schema`、`movie_state`），以便你已经写对的部分继续对照。答题单元、理解题和隔天复习都留给你填写。

## 0. 全局地图与时间线

### 0.1 从"想让模型好好回答"到"程序能拿它写代码"，中间隔着八个时刻

```text
T1 定义契约        你写下字段名和规则                     第2节
T2 导出说明书      把契约变成 JSON Schema                  第3节
T3 装进请求        说明书写进提示词或输出模式              第3.1节
        ↓  请求发给模型
T4 模型回答文本    一段字符串，可能被拒绝、截断或加围栏
        ↓  程序接手
T5 检查拒绝        明确拒绝就到此为止                      第4节
T6 解析 JSON       文本变成 dict                           第1节
T7 校验字段        dict 变成合格对象                       第2节
        ↓
T8 业务使用        record.model_dump()["rating"]
```

这张图的用法：T1～T3 发生在请求发出之前，T5～T7 发生在回答回来之后。两段可以分开调试，出问题时先判断它属于哪一段，再决定去看哪一节。

### 0.2 每个知识点的出场时刻

| 知识点 | 出场时刻 | 本版章节 |
|---|---|---|
| `json.loads` / `json.dumps` | T6 | 第1节 |
| Pydantic 模型、`model_validate` / `model_dump` | T7 | 第2节 |
| 必填、可空、默认值 | T1 | 第2.1节 |
| 枚举、范围、禁止额外字段 | T1 | 第2.2节 |
| `model_json_schema` | T2 | 第3节 |
| `response_format` 与结构化输出 | T3 | 第3.1节 |
| 拒绝、空正文、语法失败、字段失败 | T4、T5 | 第4节 |

### 0.3 本课最容易在"时刻"上出错的四件事

1. **收什么决定用哪个函数**：`json.loads` 收字符串，`json.load` 收文件对象；
2. **判断顺序**：先判断 `content is None`，再调用 `content.strip()`；顺序写反会抛出 `AttributeError`；
3. **对象归属**：第2节的课堂练习对象是观影记录，不是讲义示例的图书；把 `BookRecord` 用在观影数据上，校验必然失败；
4. **异常归属**：语法错抛出 `json.JSONDecodeError`，字段错抛出 `ValidationError`，两类要分开捕获。

这四条先记住位置，正文会把它们各自落到具体时刻。

## 运行本课前：安装必要依赖

要求Python 3.10或以上。终端从仓库根目录执行以下命令（PowerShell、macOS终端均可）：

```powershell
cd "03_大模型应用开发/03_结构化输出与数据校验"
python -m pip install -r requirements.txt
python -c "import pydantic, pytest; print(pydantic.__version__, pytest.__version__)"
```

`python`选择解释器；`-m pip`用该解释器运行包管理工具；`install`安装包；`-r`读取依赖清单。Pydantic负责数据规则，pytest运行作业测试；json和typing来自Python标准库，无需安装。

在Notebook中安装时，先确认当前工作目录是本课目录，再运行 `%pip install -r requirements.txt`。`%pip`使用当前Notebook内核对应的环境；安装后可能需要重启内核，从头运行示例。终端与Notebook如果使用不同解释器，终端安装成功不代表Notebook可导入。

先运行讲解示例，再填写对应答题单元，紧接着运行自动检测。初始未作答提示是正常的。

## 1. 时刻 T6：解析——JSON文本、字典和解析

JSON是一种交换数据的文本格式；Python字典是程序运行时的对象。看起来相似，使用方式不同。解析就是按JSON语法把文本读成Python对象，序列化则反过来。

例如 `'{"title":"海边","author":null}'` 是字符串，最外层引号属于Python。解析后才可以通过 `data["title"]` 读取字段。JSON对象变成dict，数组变成list，null变成None，true/false变成True/False。JSON键与字符串必须用双引号，不能写Python的None。

下面的title、author是本课开发者自定义的业务字段，不是Python关键字，也不是模型平台固定字段。它们处于模型回答内容里，由模型依据用户提供的原文提取，服务器不会自动生成作者。

`json.loads(text)`：标准库json中的函数，位置参数text是JSON字符串，返回解析后的对象；不合法时抛出 `json.JSONDecodeError`。它只读语法，不检查你想要哪些字段。
`json.dumps(data, ensure_ascii=False)`：data是待序列化对象；关键字参数ensure_ascii=False使中文原样显示；返回str。以下调用都在本地执行。

### 先预测，再运行

下面这段代码还没运行。先写出你的预测，再运行代码看结果。

```python
book_text = '{"title":"海边","author":null}'
book_data = json.loads(book_text)
print(type(book_text).__name__, type(book_data).__name__)
print(book_data["title"], book_data["author"])
print(json.loads('["海边", "山间"]'))
```

要预测四件事：`book_text` 的类型名、`book_data` 的类型名、`book_data["author"]` 的值、`json.loads('["海边", "山间"]')` 的返回值。

把预测写进下一个答题单元，然后运行示例单元，最后运行检测单元。**答案必须先从脑子里出来**，只在心里猜一下就往下看，不构成检索练习。

In [1]:
# 不看运行结果，先填写你的预测。占位符 "?" 表示还没有作答。
predict_text_type = "str"     # book_text 的 type().__name__
predict_data_type = "dict"     # book_data 的 type().__name__
predict_author = None        # book_data["author"] 的值
predict_loads_list = ["海边", "山间"]    # json.loads('["海边", "山间"]') 的返回值

In [2]:
import json

book_text = '{"title":"海边","author":null}'
book_data = json.loads(book_text)
print(type(book_text).__name__, type(book_data).__name__)  # str dict
print(book_data["title"], book_data["author"])  # 海边 None
print(json.dumps(book_data, ensure_ascii=False))
# 变化：合法JSON也可能是列表，而不是你期待的字典
print(json.loads('["海边", "山间"]'))
# 错误：单引号不是JSON字符串的引号
try:
    json.loads("{'title': '海边'}")
except json.JSONDecodeError:
    print("语法错误：这不是合法JSON")
# 边界：语法合法不表示业务字段齐全
print(json.loads('{"unexpected": 1}'))

str dict
海边 None
{"title": "海边", "author": null}
['海边', '山间']
语法错误：这不是合法JSON
{'unexpected': 1}


In [11]:
try:
    assert "predict_text_type" in globals(), "请先运行预测答题单元"
    _check_answers = (predict_text_type, predict_data_type, predict_author, predict_loads_list)
    if all(_check_value == "?" for _check_value in _check_answers):
        print("❌ 先预测：预测答题单元还是初始的 \"?\"，请先写下四个预测再运行检测")
    else:
        _check_rules = (
            ("book_text 的类型名", "str", predict_text_type),
            ("book_data 的类型名", "dict", predict_data_type),
            ("book_data[\"author\"] 的值", None, predict_author),
            ("json.loads 列表文本的返回值", ["海边", "山间"], predict_loads_list),
        )
        _check_wrong = [(_check_name, _check_want, _check_got)
                        for _check_name, _check_want, _check_got in _check_rules
                        if _check_got != _check_want]
        if not _check_wrong:
            print("✅ 先预测：四项预测全部正确")
        else:
            for _check_name, _check_want, _check_got in _check_wrong:
                print(f"❌ 先预测：{_check_name} 应为 {_check_want!r}，你预测的是 {_check_got!r}")
            print("  预测错的位置就是需要重读第1节的位置；先自己回看，再看折叠答案。")
except Exception as _check_error:
    print("❌ 先预测：", type(_check_error).__name__, str(_check_error))

✅ 先预测：四项预测全部正确


执行顺序是：接收字符串 → JSON解析 → 得到对象。没有任何一步自动确认title存在。`split(":")`之类的字符串切割遇到内容自带冒号、字段换序或嵌套就容易出错；JSON解析器理解格式本身。也不要用eval执行外部文本。

### 课堂练习1：读取观影记录（约3分钟）

目标：区分JSON文本与字典。实现 `read_movie(text: str) -> dict`。输入保证是表示JSON对象的字符串；原样返回解析结果，不改字段、不增加清理规则。语法无效时保留JSONDecodeError。

输入例子：`'{"name":"星河","score":8,"note":null}'`。
预期输出：`{"name": "星河", "score": 8, "note": None}`，类型dict。
只使用本节的json知识。下面的函数名是检测接口。

In [3]:
def read_movie(text: str) -> dict:
    return json.loads(text)

In [13]:
try:
    assert "read_movie" in globals(), "请先运行练习1的答题单元"
    _check_result = read_movie('{"name":"星河","score":8,"note":null}')
    assert isinstance(_check_result, dict), "返回值应为字典，不是JSON字符串"
    assert _check_result == {"name": "星河", "score": 8, "note": None}, "字段或null转换不正确"
    assert read_movie('{"name":"远山","score":6,"note":"重看"}')["score"] == 6, "不要写死示例答案"
    try:
        read_movie("{bad}")
    except json.JSONDecodeError:
        pass
    else:
        raise AssertionError("无效JSON应保留JSONDecodeError")
    print("✅ 课堂练习1通过")
except NotImplementedError:
    print("❌ 课堂练习1尚未完成")
except Exception as _check_error:
    print("❌ 课堂练习1：", str(_check_error))

✅ 课堂练习1通过


<details>
<summary>完成后查看参考答案与解释</summary>

```python
def read_movie(text: str) -> dict:
    return json.loads(text)
```

解析器负责语法和null转换，函数不应该写死某一部电影。

</details>

## 2. 时刻 T1 与 T7：用 Pydantic 定义「合格数据」

第二阶段用过Pydantic验证HTTP数据，现在同样用它验证模型回答。这里的“数据模型”是描述字段的Python类，不是大语言模型。

`BaseModel`是pydantic库提供的类。写 `class Book(BaseModel)` 表示Book继承它：Book因此获得验证数据、生成Schema、导出字典的能力。普通类的类型标注通常不会自动检查赋值。下面先对比，暂时只要求title是字符串。

`Book.model_validate(data)`在本地接收一个位置参数data（这里是字典），返回Book实例，失败抛出Pydantic的 `ValidationError`。`item.model_dump()`无参数，返回字段字典。两者一个“检查并创建对象”，一个“导出已有对象”。

In [4]:
from pydantic import BaseModel, ValidationError

class PlainBook:
    title: str

plain = PlainBook()
plain.title = 123
print(plain.title)  # 普通类型标注没有阻止赋值

class Book(BaseModel):
    title: str

item = Book.model_validate({"title": "海边"})
print(item.title)
print(item.model_dump())
try:
    Book.model_validate({"title": 123})
except ValidationError:
    print("Pydantic拒绝不是字符串的title")

123
海边
{'title': '海边'}
Pydantic拒绝不是字符串的title


### 2.1 必填、可空、默认值是不同问题

“必填”问的是键能否缺失；“可空”问的是值能否为None。JSON的null解析后就是None。以下写法里的 `str | None` 是Python联合类型：

| 字段声明 | 省略字段 | 显式字符串 | 显式None |
|---|---|---|---|
| `author: str` | 错误 | 接受 | 错误 |
| `author: str | None` | 错误 | 接受 | 接受 |
| `author: str | None = None` | 默认None | 接受 | 接受 |
| `author: str = ""` | 默认空字符串 | 接受 | 错误 |

空字符串是一个真实的字符串，None表示没有值，两者不相同。不要把“原文未提供”随便变成空字符串或编造内容。本课选择必填且可空，让模型明确用null表达未知。

Pydantic的 `Field`是创建字段配置的函数：`Field(...)`和`Field(default=...)`中省略号均表示必填；`Field(default=None)`提供缺省值，字段类型还须允许None。本课只使用通过类型的默认值，避免依赖默认值的隐式校验。

In [5]:
class RequiredAuthor(BaseModel):
    author: str | None

class DefaultAuthor(BaseModel):
    author: str | None = None

for payload in ({}, {"author": None}, {"author": "林同学"}):
    for model in (RequiredAuthor, DefaultAuthor):
        try:
            print(model.__name__, payload, "→", model.model_validate(payload).model_dump())
        except ValidationError:
            print(model.__name__, payload, "→ 缺少必填字段")

RequiredAuthor {} → 缺少必填字段
DefaultAuthor {} → {'author': None}
RequiredAuthor {'author': None} → {'author': None}
DefaultAuthor {'author': None} → {'author': None}
RequiredAuthor {'author': '林同学'} → {'author': '林同学'}
DefaultAuthor {'author': '林同学'} → {'author': '林同学'}


### 2.2 枚举、范围与多余字段

枚举就是“只能从给定值中选一个”。标准库typing的 `Literal["paper", "ebook"]` 是类型标注，限制值为这两个字符串；不是自由填写一段文字。

`Field(ge=1, le=5)`：两个关键字参数分别设定最小值和最大值；不提供default，因此仍必填。此调用返回字段配置，不是页数或评分的实际值。
`ConfigDict`来自pydantic，用于声明模型配置；`strict=True`关闭本课字符串/整数间的自动转换，`extra="forbid"`拒绝多余字段。默认宽松模式可能把"4"转成4；我们希望检测模型是否真的输出数字。配置名model_config由Pydantic规定，不是业务字段。

示例字段契约如下。所有字段名由本课开发者定义，都不是Python关键字；原文由用户提交，以下字段值由模型提取到回答JSON中，服务器只验证、不补编。

| 字段 | 类型与要求 | 省略 | 示例 |
|---|---|---|---|
| title | str | 拒绝 | "海边" |
| medium | "paper"或"ebook" | 拒绝 | "paper" |
| rating | int，1～5 | 拒绝 | 4 |
| author | str或None | 拒绝 | None |

从dict到字段转换或拒绝，是调用model_validate时Pydantic内部完成的；用户代码没有每个字段对应的if判断。

In [6]:
from typing import Literal
from pydantic import ConfigDict, Field

class BookRecord(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")
    title: str
    medium: Literal["paper", "ebook"]
    rating: int = Field(ge=1, le=5)
    author: str | None

valid_book = {"title": "海边", "medium": "paper", "rating": 4, "author": None}
print(BookRecord.model_validate(valid_book).model_dump())
print(BookRecord.model_validate({**valid_book, "medium": "ebook", "author": "林同学"}).model_dump())
# **字典解包：复制字段，并用后面的同名键替换值
for changes in ({"medium": "audio"}, {"rating": 6}, {"rating": "4"}, {"price": 20}):
    try:
        BookRecord.model_validate({**valid_book, **changes})
    except ValidationError as error:
        # errors()不传参数，返回错误字典列表；loc为字段位置，type为错误类别
        print([(entry["loc"], entry["type"]) for entry in error.errors()])

{'title': '海边', 'medium': 'paper', 'rating': 4, 'author': None}
{'title': '海边', 'medium': 'ebook', 'rating': 4, 'author': '林同学'}
[(('medium',), 'literal_error')]
[(('rating',), 'less_than_equal')]
[(('rating',), 'int_type')]
[(('price',), 'extra_forbidden')]


In [10]:
valid_book = {"title": "海边", "medium": "paper", "rating": 4, "author": None}
BookRecord.model_validate(valid_book).model_dump()

{'title': '海边', 'medium': 'paper', 'rating': 4, 'author': None}

上面的错误依次来自：枚举不允许、数字超过范围、严格模式拒绝字符串、多余字段。errors()中的loc和type由Pydantic定义，不是模型生成的业务字段；服务器上的校验库在失败时产生它们。loc是定位字段的元组，type是错误类别字符串。本课不要求记住错误类别拼写。

### 课堂练习2：定义观影数据模型（约8分钟）

定义类 `MovieRecord`，继承BaseModel。以下都是模型回答JSON中的字段，由你定义字段规则，值来自原始影评：

| 字段 | 类型与要求 | 是否必填 |
|---|---|---|
| name | str，影片名 | 是 |
| mood | Literal，只能"positive"或"negative" | 是 |
| score | int，0～10，不能接受字符串数字 | 是 |
| reviewer | str或None，未知时None | 是 |

拒绝额外字段。输入例子：`{"name":"星河","mood":"positive","score":8,"reviewer":None}`。调用MovieRecord.model_validate后应得到实例，model_dump应与输入相同；缺少reviewer应失败，reviewer=None应通过。这里只写模型声明，不需要手写逐个字段的if。

In [11]:
class MovieRecord(BaseModel):
    # TODO：声明本题需要的模型配置与四个字段
    model_config = ConfigDict(strict=True, extra="forbid")
    name: str
    mood: Literal["positive", "negative"]
    score: int = Field(ge=0, le=10)
    reviewer: str|None

In [12]:
try:
    assert "MovieRecord" in globals(), "请先运行练习2的答题单元"
    assert issubclass(MovieRecord, BaseModel), "MovieRecord应继承BaseModel"
    _check_base = {"name": "星河", "mood": "positive", "score": 8, "reviewer": None}
    assert MovieRecord.model_validate(_check_base).model_dump() == _check_base, "字段没有完整保留，请检查模型声明"
    for _check_score in (0, 10):
        _check_data = {**_check_base, "score": _check_score, "mood": "negative", "reviewer": "小林"}
        assert MovieRecord.model_validate(_check_data).model_dump() == _check_data, "合法边界或另一枚举应通过"
    for _check_bad in (
        {"name": "星河", "mood": "positive", "score": 8},
        {**_check_base, "mood": "neutral"},
        {**_check_base, "score": 11},
        {**_check_base, "score": -1},
        {**_check_base, "score": "8"},
        {**_check_base, "name": 8},
        {**_check_base, "reviewer": 8},
        {**_check_base, "extra": True},
    ):
        try:
            MovieRecord.model_validate(_check_bad)
        except ValidationError:
            pass
        else:
            raise AssertionError("应拒绝这个输入：" + str(_check_bad))
    print("✅ 课堂练习2通过")
except Exception as _check_error:
    print("❌ 课堂练习2：", str(_check_error))

✅ 课堂练习2通过


<details>
<summary>完成后查看参考答案与解释</summary>

```python
class MovieRecord(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")
    name: str
    mood: Literal["positive", "negative"]
    score: int = Field(ge=0, le=10)
    reviewer: str | None
```

reviewer没有默认值，所以不能省略；联合类型允许显式None。校验在调用model_validate时发生。

</details>

## 3. 时刻 T2：JSON Schema——把规则变成可以传递的说明书

Schema意为结构说明。JSON Schema用JSON描述“允许哪些字段、什么类型、哪些必填”。它描述规则，不是一次提取结果。

`BookRecord.model_json_schema()`：Pydantic模型的类方法，无参数，返回描述规则的dict，在本地生成，不请求模型。随后用json.dumps可以得到用于传递或展示的JSON文本。

| Schema键 | 含义和数据类型 |
|---|---|
| type | 类型名称字符串，例如"object"、"integer" |
| properties | 字段名到字段规则的字典 |
| required | 必填字段名列表 |
| enum | 允许值列表 |
| minimum / maximum | 数字下限/上限 |
| anyOf | 多个候选规则的列表，符合其中一种即可 |
| additionalProperties | 本课为false，表示禁止未声明字段 |

这些键来自JSON Schema规范，不是Python关键字；在本课由Pydantic根据开发者模型声明生成。如要约束远端输出，需要客户端另行把服务商支持的Schema送进请求，服务器不会因为你在本地定义了类就知道规则。

下面打印的author规则同时允许string和null，但author仍出现在required中。

In [13]:
book_schema = BookRecord.model_json_schema()
print(json.dumps(book_schema, ensure_ascii=False, indent=2))
# indent=2是json.dumps的关键字参数，使每层缩进2个空格；返回值仍是str
print("author规则：", book_schema["properties"]["author"])
print("必填字段：", book_schema["required"])
# 变化与反例：两个模型都允许None，但只有前者要求字段出现
print(RequiredAuthor.model_json_schema().get("required", []))
print(DefaultAuthor.model_json_schema().get("required", []))
# dict.get(key, default)：存在时取值，否则返回默认值；第二个参数[]是空列表

{
  "additionalProperties": false,
  "properties": {
    "title": {
      "title": "Title",
      "type": "string"
    },
    "medium": {
      "enum": [
        "paper",
        "ebook"
      ],
      "title": "Medium",
      "type": "string"
    },
    "rating": {
      "maximum": 5,
      "minimum": 1,
      "title": "Rating",
      "type": "integer"
    },
    "author": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "title": "Author"
    }
  },
  "required": [
    "title",
    "medium",
    "rating",
    "author"
  ],
  "title": "BookRecord",
  "type": "object"
}
author规则： {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'title': 'Author'}
必填字段： ['title', 'medium', 'rating', 'author']
['author']
[]


### 3.1 提示词、JSON模式、严格结构化输出

| 方式 | 作用 | 仍需注意 |
|---|---|---|
| 提示词要求JSON | 用文字告诉模型期望格式 | 可能多出解释或代码围栏 |
| JSON模式 | 约束输出为JSON | 不等于满足所有业务字段规则 |
| 支持Schema的结构化输出 | 在支持的模型/API上约束输出结构 | 拒绝、截断和业务事实仍需处理 |

[OpenAI官方结构化输出说明](https://developers.openai.com/api/docs/guides/structured-outputs)区分JSON模式和遵循Schema的输出，并说明拒绝等情况。服务商支持的Schema子集可能不同，本地生成的Schema不能不加核对就直接发送。

你上一课使用DeepSeek兼容API。[DeepSeek JSON Output文档](https://api-docs.deepseek.com/guides/json_mode/)要求使用response_format的json_object模式，提示词包含json并提供格式示例，同时提醒截断和空内容。下面只构造请求参数，不发请求。具体模型名称由你已有配置提供，这里占位。

response_format是API规定的请求字段，由客户端发送；值是dict，其中type是模式字符串。messages、model、max_tokens沿用上一课；它们不属于回答里的图书字段。把Schema写进提示词只是说明要求，不会自动启用服务端严格Schema校验。

In [14]:
json_request = {
    "model": "替换为配置中的模型名称",
    "messages": [
        {"role": "system", "content": '提取图书信息，只输出JSON，不要代码围栏。格式示例：{"title":"海边","medium":"paper","rating":4,"author":null}。未知作者用null，不编造。'},
        {"role": "user", "content": "《海边》是纸质书，我打4分，没有记录作者。"},
    ],
    "response_format": {"type": "json_object"},
    "max_tokens": 500,
}
print(json_request["response_format"])
# 这就是不满足业务规则的合法JSON：JSON模式本身不等于本地字段校验
wrong_shape = json.loads('{"title":"海边","rating":"很棒"}')
try:
    BookRecord.model_validate(wrong_shape)
except ValidationError:
    print("JSON语法通过，但业务结构未通过")

{'type': 'json_object'}
JSON语法通过，但业务结构未通过


### 课堂练习3：导出观影规则（约3分钟）

实现 `movie_schema() -> dict`，不接收参数，返回练习2的MovieRecord对应的JSON Schema。先完成练习2。
预期：根type为object；required包含四个字段；score规则含minimum=0和maximum=10；reviewer允许string或null；额外字段禁止。
目标是从模型声明导出规则，不是写一条电影数据，也不用手抄整个Schema。

In [15]:
def movie_schema() -> dict:
    # TODO：从 MovieRecord 导出 JSON Schema
    return MovieRecord.model_json_schema()

In [16]:
try:
    assert "movie_schema" in globals(), "请先运行练习3的答题单元"
    _check_schema = movie_schema()
    assert isinstance(_check_schema, dict), "返回值应是dict"
    assert _check_schema["type"] == "object", "根类型应是object"
    assert set(_check_schema["required"]) == {"name", "mood", "score", "reviewer"}, "四个字段都必填"
    assert _check_schema["additionalProperties"] is False, "应禁止额外字段"
    assert _check_schema["properties"]["score"]["minimum"] == 0, "score下限应为0"
    assert _check_schema["properties"]["score"]["maximum"] == 10, "score上限应为10"
    assert set(_check_schema["properties"]["mood"]["enum"]) == {"positive", "negative"}, "枚举不正确"
    assert {x["type"] for x in _check_schema["properties"]["reviewer"]["anyOf"]} == {"string", "null"}, "reviewer应可空"
    assert _check_schema == MovieRecord.model_json_schema(), "规则应来自MovieRecord"
    print("✅ 课堂练习3通过")
except NotImplementedError:
    print("❌ 课堂练习3尚未完成")
except Exception as _check_error:
    print("❌ 课堂练习3：", str(_check_error))

✅ 课堂练习3通过


<details>
<summary>完成后查看参考答案与解释</summary>

```python
def movie_schema() -> dict:
    return MovieRecord.model_json_schema()
```

Schema描述所有合法记录，不是某一条记录的内容。题目依赖练习2的模型声明。

</details>

## 4. 时刻 T4 与 T5：接到回答之后

原始模型回答属于外部数据。程序按顺序处理：检查拒绝 → 解析JSON → 校验字段 → 才进入业务逻辑。拒绝不是“缺少几个字段”，不能填一份假数据当成功。

为了离线练习，我们约定应用接收两个参数：content是str或None，表示模型回答文本；refusal是str或None，非空字符串表示明确拒绝。这两个参数名由本课应用定义，不是Python关键字。真实对接时，由应用根据服务商的响应提取；不能假设所有兼容API都提供相同的拒绝字段。没有明确拒绝信号时，也不要把所有解析失败都猜成拒绝。

下面示例 `read_book_content` 是我们自己定义的函数，接收content，成功返回BookRecord实例；语法错会抛JSONDecodeError，字段错抛ValidationError。这样能把两种失败分开观察。


In [17]:
def read_book_content(content: str) -> BookRecord:
    data = json.loads(content)
    return BookRecord.model_validate(data)

for sample in (
    '{"title":"海边","medium":"paper","rating":4,"author":null}',
    '{"title":',  # 截断的JSON
    '{"title":"海边","medium":"paper","rating":9,"author":null}',
):
    try:
        print("成功：", read_book_content(sample).model_dump())
    except json.JSONDecodeError:
        print("语法失败：文本不能完整解析")
    except ValidationError:
        print("校验失败：JSON可以解析，但字段不合格")

成功： {'title': '海边', 'medium': 'paper', 'rating': 4, 'author': None}
语法失败：文本不能完整解析
校验失败：JSON可以解析，但字段不合格


### 4.1 把失败变成明确的应用结果

下面 `book_state(content, refusal=None)` 返回状态字符串。refusal默认None，省略表示没有明确拒绝信号；显式None含义相同，传入非空字符串则优先判为拒绝。content没有默认值，调用者必须传入，但可以传None表示没有正文。

这些状态是应用自行约定的字符串，不是HTTP状态码，也不是服务商原样返回的字段。拒绝优先于content：即使正文碰巧能解析，只要明确拒绝，就不能继续当成功。

空正文、JSON语法错误和字段错误都应阻止业务使用。使用具体异常类型可以只捕获预期的数据问题，不把拼错变量等程序错误吞掉。

In [ ]:
def book_state(content: str | None, refusal: str | None = None) -> str:
    if refusal:
        return "refused"
    if content is None or not content.strip():
        return "empty"
    try:
        read_book_content(content)
    except json.JSONDecodeError:
        return "bad_json"
    except ValidationError:
        return "bad_fields"
    return "ok"

print(book_state(json.dumps(valid_book)))  # ok
print(book_state(None))  # empty
print(book_state("   "))  # empty
print(book_state("{"))  # bad_json
print(book_state("{}"))  # bad_fields
print(book_state(json.dumps(valid_book), "无法提供"))  # refused
# strip()是str的方法，无参数时去掉两端空白，返回新str；这里仅用于识别空正文

实际接口还应先查看生成是否完整：若输出因长度限制而截断，不能当作完整结果使用。本课将截断JSON作为解析失败样例，不实现接口适配层或自动重试。Markdown围栏包着的JSON也不是纯JSON；本课直接拒绝，避免“截一段看起来像JSON”掩盖问题。

结构正确也不等于事实正确：如果原文说作者未知，模型却给出一个虚构姓名，这个姓名依然是合法str。Pydantic不能自动证明内容来自原文，应用还应与来源核对。课堂检测能验证程序行为，不能保证真实模型每次提取都准确。

### 课堂练习4：只在数据合格时放行（约8分钟）

实现 `movie_state(content: str | None, refusal: str | None = None) -> str`。依赖练习2的MovieRecord。输入来自一次回答，不是原始影评。

| 输入情况 | 返回值 |
|---|---|
| refusal为非空字符串，无论content是什么 | "refused" |
| 无拒绝，content是None、空串或纯空白 | "empty" |
| 非空正文不是合法JSON，包括代码围栏 | "bad_json" |
| JSON合法，但MovieRecord校验失败 | "bad_fields" |
| JSON与字段均合格 | "ok" |

具体例子：`movie_state('{"name":"星河","mood":"positive","score":8,"reviewer":null}')`返回"ok"；
`movie_state("{}")`返回"bad_fields"；
`movie_state(None, "无法提供")`返回"refused"。
不联网、不返回伪造的成功记录、不要求重试。

In [19]:
def movie_state(content: str | None, refusal: str | None = None) -> str:
    # TODO：按讲义第4节的顺序识别拒绝、空正文、语法失败和字段失败
    if refusal:
        return 'refused'
    if content is None or not content.strip():
        return "empty"
    try:
        MovieRecord.model_validate(json.loads(content))
    except json.JSONDecodeError:
        return "bad_json"
    except ValidationError:
        return 'bad_fields'

    return 'ok'

    

In [20]:
try:
    assert "movie_state" in globals(), "请先运行练习4的答题单元"
    for _check_content, _check_refusal, _check_expected in (
        ('{"name":"星河","mood":"positive","score":8,"reviewer":null}', None, "ok"),
        ('{"name":"远山","mood":"negative","score":0,"reviewer":"小林"}', None, "ok"),
        (None, "拒绝", "refused"),
        ('{}', "拒绝", "refused"),
        (None, None, "empty"),
        ("  ", None, "empty"),
        ("", None, "empty"),
        ("{", None, "bad_json"),
        ('```json\n{}\n```', None, "bad_json"),
        ("{}", None, "bad_fields"),
        ("[]", None, "bad_fields"),
        ('{"name":"星河","mood":"positive","score":11,"reviewer":null}', None, "bad_fields"),
    ):
        _check_actual = movie_state(_check_content, _check_refusal)
        assert _check_actual == _check_expected, f"输入{_check_content!r}、拒绝{_check_refusal!r}应返回{_check_expected}，实际{_check_actual!r}"
    print("✅ 课堂练习4通过")
except NotImplementedError:
    print("❌ 课堂练习4尚未完成")
except Exception as _check_error:
    print("❌ 课堂练习4：", str(_check_error))

✅ 课堂练习4通过


<details>
<summary>完成后查看参考答案与解释</summary>

```python
def movie_state(content: str | None, refusal: str | None = None) -> str:
    if refusal:
        return "refused"
    if content is None or not content.strip():
        return "empty"
    try:
        MovieRecord.model_validate(json.loads(content))
    except json.JSONDecodeError:
        return "bad_json"
    except ValidationError:
        return "bad_fields"
    return "ok"
```

JSON解析与字段校验是两道检查。把它们都写成成功，会让无效数据进入业务。

</details>

## 5. 完整流程回顾

```text
开发者定义字段契约 → Pydantic模型 → JSON Schema规则
                                     ↓
用户原文 → 提取提示词/受支持的输出模式 → 模型回答文本
                                     ↓
                              检查拒绝与完整性
                                     ↓
                              JSON解析 → 字段校验
                                     ↓
                       合格数据进入业务；失败单独处理
```

本课已经能用固定文本验证流程，下一课再学习多轮对话和流式响应。不要把还没接收完的流式片段当成本课的完整JSON文本。

把已验证实例导出字典后，业务可以访问字段。例如 `record.model_dump()["rating"]`，不需要从一句自然语言中切数字。

## 6. 课末作业：客服工单结构化提取

### 背景与允许修改范围

原始反馈例如：“导出时闪退，请尽快处理，联系我：小陈。”模型应提取一份工单。你的工作是定义接收规则、导出规则、接收并验证模型回答；不需要自己写自然语言提取算法，也不需要调用真实API。

只修改 `structured_ticket.py` 的三个TODO函数体；保留函数名和参数，不修改测试文件。本题用函数返回模型类，是为了让尚未完成的题目也能正常导入：例如 `Ticket = create_ticket_model()` 后，可使用 `Ticket.model_validate(data)`。Python中类本身也是可以返回的对象。它返回类，不是实例或字典。

### 任务1：create_ticket_model() → type[BaseModel]

无输入，返回你定义的Pydantic模型类，类名可自行选择。所有字段必须出现、禁止额外字段、启用严格类型校验。

| 模型输出字段 | 精确类型与规则 | 示例 |
|---|---|---|
| summary | str，反馈摘要 | "导出闪退" |
| category | Literal["bug", "question"]，故障或咨询 | "bug" |
| priority | int，1～3 | 3 |
| contact | str或None，原文未提供时None | "小陈" |

这些字段名由本作业定义，不是Python关键字。字段值由模型从用户原文提取，应用不会生成联系人。空字符串允许，不要求清理空格；重点是声明结构。
输入示例：`{"summary":"导出闪退","category":"bug","priority":3,"contact":None}`。校验后实例的model_dump应原样返回该字典。缺少contact拒绝，显式None通过；priority="3"拒绝，1和3都通过。

### 任务2：ticket_schema() → dict

无输入，返回任务1模型类生成的JSON Schema。应包含四个字段的规则、required、禁止额外字段，以及priority的上下限。返回规则字典，不是JSON字符串或某一条工单。

### 任务3：parse_ticket_result(content, refusal=None) → dict

content是str或None，表示完整的模型回答正文；refusal是str或None，表示应用从接口整理出的明确拒绝原因，默认None。测试不传其他类型；空refusal字符串按无拒绝处理。

返回固定两个键：status（str）和data（dict或None）。它们是本作业定义的应用结果字段，均由本地函数生成，不由模型填写，也不是HTTP响应契约。

| 情况 | 完整返回格式 |
|---|---|
| refusal非空，优先于正文 | {"status": "refused", "data": None} |
| 无拒绝且正文None、空串、空白 | {"status": "empty", "data": None} |
| 非法JSON、截断、代码围栏 | {"status": "invalid_json", "data": None} |
| JSON合法但字段不符、顶层数组等 | {"status": "invalid_data", "data": None} |
| 合格工单 | {"status": "ok", "data": 校验后的字段字典} |

输入例子：

```python
parse_ticket_result('{"summary":"导出闪退","category":"bug","priority":3,"contact":null}')
# 应返回：
# {"status":"ok","data":{"summary":"导出闪退","category":"bug","priority":3,"contact":None}}

parse_ticket_result('{"summary":"导出闪退"}')
# {"status":"invalid_data","data":None}
parse_ticket_result(None, "无法提供")
# {"status":"refused","data":None}
```

不需要自行修复JSON或重试；错误时data必须是None，不能填默认工单。任务3依赖任务1；先完成模型，再测试流程。

### 学习目标与测试

| 核心目标 | 作业任务 | 测试内容 |
|---|---|---|
| 声明必填、枚举、可空与范围 | 任务1 | 合法工单、缺字段、非法枚举/范围/类型、多余字段 |
| 从同一模型导出Schema | 任务2 | 字段、required、枚举、可空、范围、额外字段 |
| 解析校验并阻止无效输出 | 任务3 | 成功、拒绝优先、空内容、语法失败、校验失败 |

在本课目录运行：

```powershell
python -m pytest -q test_structured_ticket.py
```

共19项测试，全部对应本课数据契约与输出处理；没有单独考查通用字符串清理的题目。初始骨架的19项失败应全部源于TODO的NotImplementedError。可分别运行 `python -m pytest -q test_structured_ticket.py -k model`、`-k schema`、`-k parse` 检查三部分。自动测试验证结构契约，不能代替真实提取质量的评估。

## 7. 理解题

先凭记忆作答，再运行检测。这是本课的第二次检索练习。

每题只有一个正确答案，用字母作答。作答之后运行紧下方的检测单元；检测单元只会解释你答错的题。

**第1题** 调用 `json.loads(text)` 之后得到的是什么？

- A 原样返回的文本副本
- B 解析得到的字典对象
- C 字段校验失败的错误
- D 可读取该文本的句柄

**第2题** 为什么 `json.loads` 成功之后还要做 Pydantic 校验？

- A 解析器无法识别双引号语法
- B 校验可把字符串转为数字值
- C 校验能够证明内容来自原文
- D 解析只保证语法不保证字段

**第3题** `author: str | None` 与 `author: str | None = None` 的区别是什么？

- A 前者省略字段失败后者不失败
- B 前者拒绝空值后者接受空值
- C 前者要求字符串后者要求空值
- D 两者在省略字段时都会失败

**第4题** `Literal`、`Field`、`ConfigDict` 在本课分别负责什么？

- A 数值范围、模型配置、枚举取值
- B 字段名称、数值范围、模型配置
- C 枚举取值、数值范围、模型配置
- D 模型配置、枚举取值、数值范围

**第5题** 在本地用 `model_json_schema` 生成 Schema，会约束远端模型的输出吗？

- A 会，本地定义类之后即自动生效
- B 不会，必须把规则送进请求才生效
- C 会，字段名与服务商一致即生效
- D 不会，Schema并非请求参数

**第6题** 拒绝、空正文、语法失败与字段失败为什么要分开处理？

- A 分开可以让错误信息更简短
- B 分开能减少需要编写的代码量
- C 不分开就无法通过类型标注
- D 四种情况对应不同修复动作

**第7题** 字段校验通过，能证明信息来自原文吗？

- A 不能，虚构姓名也是合法字符串
- B 能，校验过程已经确认字段类型
- C 能，只要模型没有给出拒绝信号
- D 不能，但是拒绝时可以用来判断

In [ ]:
# 用字母作答，未作答的位置留空字符串。
answers = {
    "q1": "B",
    "q2": "D",
    "q3": "A",
    "q4": "C",
    "q5": "B",
    "q6": "D",
    "q7": "A",
}

In [ ]:
_key = {"q1": "B", "q2": "D", "q3": "A", "q4": "C", "q5": "B", "q6": "D", "q7": "A"}
_explain = {
    "q1": "json.loads 接收字符串并返回解析后的对象；收文件对象的那个函数是 json.load。",
    "q2": "解析只检查 JSON 语法，字段名、类型、范围和必填由模型声明负责。",
    "q3": "两种写法都可空，区别在于省略字段时：没有默认值就失败，有默认值则填入 None。",
    "q4": "Literal 限制枚举取值，Field 设定数值范围，ConfigDict 声明模型级配置。",
    "q5": "本地生成的 Schema 只是规则文本，必须由客户端送进请求才可能约束远端输出。",
    "q6": "四种情况的修复动作不同：重新请求、重发内容、修语法、改契约或改提示词。",
    "q7": "Pydantic 只验证结构，无法证明值来自原文，虚构的姓名同样是合法字符串。",
}
try:
    assert "answers" in globals(), "请先运行理解题答题单元"
    _check_bad = [k for k, v in answers.items() if str(v).strip() and str(v).strip().upper() not in "ABCD"]
    _check_blank = [k for k, v in answers.items() if not str(v).strip()]
    if _check_bad:
        print("❌ 理解题：这些题的答案不是 A/B/C/D 中的单个字母：", ", ".join(sorted(_check_bad)))
    elif _check_blank:
        print("❌ 理解题：这些题还没有作答：", ", ".join(sorted(_check_blank)))
    else:
        _check_wrong = [k for k in sorted(_key) if str(answers[k]).strip().upper() != _key[k]]
        if not _check_wrong:
            print("✅ 理解题：7题全部正确")
        else:
            for _check_q in _check_wrong:
                print(f"❌ {_check_q}：你选的是 {str(answers[_check_q]).strip().upper()}，{_explain[_check_q]}")
            print("  其余题目已正确。错题对应的章节见第0.2节的出场时刻表。")
except Exception as _check_error:
    print("❌ 理解题：", type(_check_error).__name__, str(_check_error))

## 8. 隔天复习（检索练习）

**明天**再打开这份讲义做这一节，不是今天做完就顺手做掉。间隔本身就是练习的一部分。

做法：**不要回看上面的任何单元**，拿出一张白纸或新建一个单元，凭记忆写出下面四项，写完再回来对照。

1. 画出八个时刻的时间线：从"定义契约"到"业务使用"，标出模型在哪个时刻介入；
2. 写出五个调用的输入与输出：`json.loads`、`json.dumps`、`model_validate`、`model_dump`、`model_json_schema`；
3. 写出收到回答之后的判断顺序，以及四种失败各自对应的返回值；
4. 不看讲义，重写 `read_movie`、`MovieRecord`、`movie_schema`、`movie_state` 四个接口。

写不出来的位置就是真实缺口。回到第0.2节的出场时刻表找到对应章节重读那一节，不要从头再看一遍。

注意：运行过、看过、点头过都只是覆盖，不构成学习。只有在没有提示的情况下能重新写出来才算。

第二天重写完成后，把结果和这一节对照，并在 `课程设计与验收.md` 里补一行客观结果（哪几项写出来、哪几项没写出来）。

## 9. 速查材料

本课已产出 `速查_结构化输出.md`，与本讲义放在同一目录。

讲义用来看懂因果和过程，速查材料用来三个月后直接上手写代码。它只保留结论和索引，不重复本讲的解释过程。

速查材料包含四类内容：

- 五个调用的语法与返回值对照；
- 必填、可空、默认值的四种组合行为表；
- 收到回答之后的判断顺序与四种失败返回值；
- 排错索引：某个异常对应什么问题、先去查哪个属性。

如果后续课程引入新的状态名或字段名，必须沿用本课已有的这一套说法，不要为同一个概念换名字。

## 参考资料

核对日期：2026-09-18。本课本地行为在Pydantic 2.x下验证；平台参数以实际使用的模型与服务商文档为准。

- [Pydantic模型与验证](https://docs.pydantic.dev/latest/concepts/models/)
- [Pydantic字段、默认值和约束](https://docs.pydantic.dev/latest/concepts/fields/)
- [OpenAI结构化输出与JSON模式](https://developers.openai.com/api/docs/guides/structured-outputs)
- [DeepSeek JSON Output](https://api-docs.deepseek.com/guides/json_mode/)
